In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("../data/raw/dataset.csv")
df = data.copy()
# changer les types des columns : 

df["Order ID"] = df["Order ID"].astype("string")
df["Ship Mode"] = df["Ship Mode"].astype("string")
df["Customer ID"] = df["Customer ID"].astype("string")
df["Customer Name"] = df["Customer Name"].astype("string")
df["Segment"] = df["Segment"].astype("string")
df["Country"] = df["Country"].astype("string")
df["City"] = df["City"].astype("string")
df["State"] = df["State"].astype("string")
df["Region"] = df["Region"].astype("string")
df["Product ID"] = df["Product ID"].astype("string")
df["Category"] = df["Category"].astype("string")
df["Sub-Category"] = df["Sub-Category"].astype("string")
df["Product Name"] = df["Product Name"].astype("string")
df["Sales"] = df["Sales"].astype("float")
# converte apres
df["Quantity"] = pd.to_numeric(df["Quantity"],errors='coerce')
# print(df["Quantity"].dtype)

df["Discount"] = df["Discount"].astype("float")
df["Profit"] = df["Profit"].astype("float")

df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
print(df.dtypes)


Row ID                    int64
Order ID         string[python]
Order Date       datetime64[ns]
Ship Date                object
Ship Mode        string[python]
Customer ID      string[python]
Customer Name    string[python]
Segment          string[python]
Country          string[python]
City             string[python]
State            string[python]
Postal Code              object
Region           string[python]
Product ID       string[python]
Category         string[python]
Sub-Category     string[python]
Product Name     string[python]
Sales                   float64
Quantity                float64
Discount                float64
Profit                  float64
dtype: object


In [2]:
# converte les valeus des colonnes (Customer Name,Category,Segment,City) par meme formate et repmlir les noms manquant de product name:
# ----------------------------------------------------------------------------------

df["Customer Name"] = df["Customer Name"].str.strip().str.title()
df["Category"] = df["Category"].str.strip().str.title()
df["Segment"] = df["Segment"].str.strip().str.title()
df["State"] = df["State"].str.strip().str.title()
df["City"] = df["City"].str.strip().str.title()

df.groupby("Product ID")["Product Name"].transform(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan )

new_segment = {"Home Ofice" : "Home Office", "Consumerr" : "Consumer", "Corporrate" : "Corporate"}
df["Segment"] = df["Segment"].replace(new_segment)


In [3]:
# remplir les valeurs manquant par des noms des customers par id customer :
# -------------------------------------------------------------------------

# print(df.groupby("Customer ID")["Customer Name"].nunique())
print(df["Customer Name"].isna().sum())

df["Customer Name"] = df.groupby("Customer ID")["Customer Name"].transform(lambda x: x.ffill().bfill())
print(df["Customer Name"].isna().sum())

df = df.dropna(subset=["Customer Name"])
# df[df["Customer Name"].isna()]
print(df["Customer Name"].isna().sum())



351
1
0


In [4]:
# sepprission des doublent :

print("duplicate :",df.duplicated().sum())
df = df.drop_duplicates()
print("duplicate :",df.duplicated().sum())



duplicate : 59
duplicate : 0


In [5]:
# remplir les valeur manquant par des code postal par city :
# ---------------------------------------------------------

print(df["Postal Code"].isna().sum())

df["Postal Code"] = df.groupby(["City"])["Postal Code"].transform(lambda x: x.ffill().bfill())

# df.dropna(subset=['Postal Code'], inplace=True)
print(df["Postal Code"].isna().sum())




199
0


In [6]:
# supression des valeurs manquant de quantity :
# --------------------------------------------

# df['Quantity'].isna().sum()
# print((df['Quantity'] < 0).sum())

df = df[df['Quantity'] >= 0]





In [7]:
# supression les valeurs plus de 1 dans colonne Discount :
# -------------------------------------------------------

df[df['Discount'] > 1]
df = df[df['Discount'] <= 1]
df[df['Discount'] > 1]


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit


In [8]:
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.duplicated().sum())

0
0


In [9]:
# remplire des valeurs manquant de colonne ship mode par order date :
df["Ship Mode"] = df.groupby(["Order ID"])["Ship Mode"].transform(lambda x: x.ffill().bfill())

In [ ]:
# dans cette  partie j'ai remplir des valeurs manquant des colonnes : "ship date" par "ship mode" et des valeurs manquants de ship mode par "ship date" et "order date":
# ------------------------------------#
df['Order Date'] = pd.to_datetime(df['Order Date'],format='%m/%d/%Y', errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'],format='%m/%d/%Y', errors='coerce')
# print(df['Ship Date'].dtype)
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# df[df['Shipping Days'] <0]
# print(df['Ship Mode'].isna().sum())
# print((df['Shipping Days'] < 0).sum())
# print(df['Ship Mode'].value_counts()
# print(df.groupby('Ship Mode')['Shipping Days'].describe()

valid_shipping = df[(df['Shipping Days'] >= 0) & (df['Shipping Days'] <= 30) & (df['Ship Mode'].notna())].copy()
shipping_days_by_mode = valid_shipping.groupby('Ship Mode')['Shipping Days'].median()

# print(valid_shipping)
# print(shipping_days_by_mode)

def fill_ship_date(row):

    if pd.notna(row['Ship Date']):
        return row['Ship Date']

    if pd.isna(row['Order Date']) or pd.isna(row['Ship Mode']):
        return row['Ship Date']

    if row['Ship Mode'] == 'Same Day':
        return row['Order Date']

    return row['Ship Date']

df['Ship Date'] = df.apply(fill_ship_date, axis=1)
# print(df['Ship Date'].isna().sum())

# print(df.groupby('Ship Mode')['Shipping Days'].value_counts().sort_index())
# print(df.groupby('Ship Mode')['Shipping Days'].median())

def get_ship_mode(days):

    if pd.isna(days) or days < 0 or days > 30:
        return np.nan

    if days == 0:
        return 'Same Day'
    elif days <= 2:
        return 'First Class'
    elif days <= 3:
        return 'Second Class'
    else:
        return 'Standard Class'


mask = df['Ship Mode'].isna()

df.loc[mask, 'Ship Mode'] = (df.loc[mask, 'Shipping Days'].apply(get_ship_mode))


print(df["Ship Mode"].isna().sum())
df.dropna(subset=['Ship Mode'],inplace=True)
print(df["Ship Mode"].isna().sum())
print(df["Ship Date"].isna().sum())
df.dropna(subset=['Ship Date'],inplace=True)
print(df["Ship Date"].isna().sum())
df = df.dropna(subset=['Order Date'])
df = df.dropna(subset=['Sales'])
df = df.drop(columns=["Shipping Days"])
print(df["Order Date"].isna().sum())






       Row ID        Order ID Order Date  Ship Date       Ship Mode  \
0           1  CA-2016-152156 2016-11-08 2016-11-11    Second Class   
1           2  CA-2016-152156 2016-11-08 2016-11-11    Second Class   
2           3  CA-2016-138688 2016-06-12 2016-06-16    Second Class   
3           4  US-2015-108966 2015-10-11 2015-10-18  Standard Class   
4           5  US-2015-108966 2015-10-11 2015-10-18  Standard Class   
...       ...             ...        ...        ...             ...   
9994     3126  CA-2015-121720 2015-06-11 2015-06-12     First Class   
10017    3051  US-2017-148054 2017-10-06 2017-10-11  Standard Class   
10034    6120  CA-2017-143378 2017-09-19 2017-09-25  Standard Class   
10046    1385  US-2016-108504 2016-02-05 2016-02-05        Same Day   
10062    4962  CA-2014-156587 2014-03-07 2014-03-08     First Class   

      Customer ID    Customer Name      Segment        Country  \
0        CG-12520      Claire Gute     Consumer  United States   
1        CG-125

In [ ]:
df["Quantity"] = df["Quantity"].astype("Int64")
df['Quantity'].dtype
print(df["Ship Date"].isna().sum())
path = "../data/processed/dataset_cleaned.csv"

df.to_csv(path, index=False)

0
